In [ ]:
# Import modules
import paramiko
import os
from dataclasses import dataclass, asdict
import argparse
import json
from datetime import datetime
from pathlib import Path
from typing import Any, Optional
from stat import S_ISDIR, S_ISREG
import pandas as pd
from abc import ABC, abstractmethod
from lxml import etree
import dotenv

In [ ]:
# Provide connection details in separate .env file for security
dotenv.load_dotenv()
hostname = os.getenv('HOSTNAME')  # Replace with your server's IP or hostname
user = os.getenv('USER')    # Replace with your SSH username    
password = os.getenv('PASSWORD')  # SSH password from .env file

PROJECT_DIRECTORY = '/mnt/data_raid/feierabend/AVL_FIRE/PEMWE/PEMStar_2'    # Project directory on the remote server
MODEL_NAME = 'PEMStar_BekaertPTL'                   # AVL FIRE model name
CASE_SET_NAME = 'PolCurve_Bek~rtPTL_Update'        # Case set name within the model
CASE_NAME = None                                    # Set to None to search all cases, or specify a case name like 'Case_1'

# Or, if you want to search for files with specific names
# TARGET_FILENAMES = ['file1.txt', 'report.log']


In [ ]:
# --- SSH Client Initialization ---
ssh_client = paramiko.SSHClient()
# Automatically add the server's host key. For production, it's better to manage known_hosts explicitly.
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the SSH server
    ssh_client.connect(hostname=hostname, username=user, password=password)
    print(f"Connected to {hostname} using password.")

    # Open an SFTP client
    sftp_client = ssh_client.open_sftp()
    print(f"Opened SFTP session.")
except paramiko.AuthenticationException:
    print("Authentication failed. Check your username, password, or private key.")
except paramiko.SSHException as e:
    print(f"Could not establish SSH connection: {e}")  

In [ ]:
simulation_project_path = f"{PROJECT_DIRECTORY}/simulation/project/"
if CASE_NAME is None:
    print("CASE_NAME is None, searching for all cases in the specified model and case set...")
    data_paths = []
    for entry in sftp_client.listdir_attr(simulation_project_path):
        if S_ISDIR(entry.st_mode) and f"{MODEL_NAME}.{CASE_SET_NAME}." in entry.filename:
            case_path = f"{simulation_project_path}{entry.filename}"
            data_path = f"{case_path}/input/{MODEL_NAME}.asix"
            data_paths.append(data_path)
            print(f"Found case: {entry.filename}, data path: {data_path}")
else:
    case_set_path = f"{simulation_project_path}{MODEL_NAME}.{CASE_SET_NAME}"
    data_path = f"{case_set_path}.{CASE_NAME}/input/{MODEL_NAME}.asix"
    print(f"Using specified CASE_NAME: {CASE_NAME}, data path: {data_path}")
    data_paths = [f"{data_path}"]


In [ ]:
KEEP_ATTRS = {
    "name",
    "index",
    "reference_id",
    "map_type",
    "data_type",
    "orientation",
}

REMAP_ATTRS = {
    "t": "type",
    "v": "value",
    "u": "unit",
}


def _cast_value(type_str: Optional[str], raw: Any) -> Any:
    """
    Cast the ASIX 'value' according to ASIX 'type'.
    If casting fails or type is unknown, returns the raw string.
    """
    if raw is None:
        return None
    if not isinstance(raw, str):
        return raw
    if not type_str:
        return raw

    t = type_str.strip().lower()
    if t == "string":
        return raw
    if t in {"int", "integer"}:
        try:
            return int(raw)
        except ValueError:
            return raw
    if t in {"double", "float", "real"}:
        try:
            return float(raw)
        except ValueError:
            return raw
    if t in {"bool", "boolean"}:
        return raw.strip().lower() in {"yes", "true", "1"}
    if t == "date":
        for fmt in ("%Y%m%d %H:%M:%S", "%Y%m%d"):
            try:
                return datetime.strptime(raw, fmt)
            except ValueError:
                pass
        return raw
    return raw


def _local_tag(el: etree._Element) -> str:
    return etree.QName(el).localname


def _try_int(s: Any) -> Optional[int]:
    if s is None:
        return None
    try:
        return int(str(s))
    except Exception:
        return None


def _sort_children_lists(node: dict) -> None:
    """
    Recursively sort lists under node["children"] by numeric attrs.index when possible.
    """
    children = node.get("children") or {}
    for v in children.values():
        if isinstance(v, list):
            idxs = [_try_int((child.get("attrs") or {}).get("index")) for child in v]
            if all(i is not None for i in idxs):
                v.sort(key=lambda child: _try_int((child.get("attrs") or {}).get("index")) or 0)
            for child in v:
                _sort_children_lists(child)
        elif isinstance(v, dict):
            _sort_children_lists(v)


def build_tree(
    root: etree._Element,
    *,
    always_list: bool = False,
    keep_extra_attrs: bool = True,
    cast_values: bool = False,
) -> dict:
    """
    Convert an lxml element into the Option A tree-first structure.
    """

    def convert(el: etree._Element) -> dict:
        node: dict = {"tag": _local_tag(el), "attrs": {}, "children": {}}

        attrs: dict = {}
        extra: dict = {}

        for k, v in el.attrib.items():
            if k in REMAP_ATTRS:
                attrs[REMAP_ATTRS[k]] = v
            elif k in KEEP_ATTRS:
                attrs[k] = v
            else:
                if keep_extra_attrs:
                    extra[k] = v

        if cast_values and "value" in attrs:
            attrs["value"] = _cast_value(attrs.get("type"), attrs.get("value"))

        node["attrs"] = attrs
        if keep_extra_attrs and extra:
            node["attrs_extra"] = extra

        # Children grouped by tag
        for child in el:
            if not isinstance(child.tag, str):
                continue
            ctag = _local_tag(child)
            cnode = convert(child)

            if ctag not in node["children"]:
                node["children"][ctag] = [cnode] if always_list else cnode
            else:
                existing = node["children"][ctag]
                if isinstance(existing, list):
                    existing.append(cnode)
                else:
                    node["children"][ctag] = [existing, cnode]

        return node

    out = convert(root)
    _sort_children_lists(out)
    return out


def parse_asix(asix_file, *, always_list: bool = False, 
               keep_extra_attrs: bool = True, cast_values: bool = True) -> dict:
    parser = etree.XMLParser(remove_comments=True, huge_tree=True)
    tree = etree.parse(asix_file, parser)
    root = tree.getroot()
    return build_tree(root, always_list=always_list, keep_extra_attrs=keep_extra_attrs, 
                      cast_values=cast_values)


def _json_default(o: Any) -> Any:
    if isinstance(o, datetime):
        return o.isoformat()
    raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")


def _count_nodes(node: dict) -> int:
    n = 1
    children = node.get("children") or {}
    for v in children.values():
        if isinstance(v, list):
            for ch in v:
                n += _count_nodes(ch)
        elif isinstance(v, dict):
            n += _count_nodes(v)
    return n

# with sftp_client.open(data_path, 'r') as data_file:
#     # data = remote_file.read()
#     data_dicts_ = parse_asix(data_file, always_list=False, keep_extra_attrs=True, cast_values=True)




In [ ]:
data_dicts_list = []
for data_path in data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        # data = remote_file.read()
        data = parse_asix(data_file, always_list=False, keep_extra_attrs=True, cast_values=True)
        data_dicts_list.append(data)
    # data_dicts
data_dicts_list

In [ ]:
# Close the connections
if 'sftp_client' in locals() and sftp_client:
    sftp_client.close()
    print("SFTP session closed.")
if 'ssh_client' in locals() and ssh_client:
    ssh_client.close()
    print("SSH connection closed.")

In [ ]:
data_dicts = data_dicts_list[0]
data_dicts

In [ ]:
for key, value in data_dicts.items():
    # if entry['reference_id'] is not None:
    print(f"{key}: {value}")

In [ ]:
with open('testing_output_v2.json', 'w', encoding='utf-8') as f:
    json.dump(data_dicts, f, default=_json_default, indent=2)